# F1 Strategy Simulator (Local OpenF1 data)

This notebook builds a **time-based strategy recommender**:

- Uses local `openf1_data/` files
- Trains a lap-time model from historical stint/lap data
- Simulates candidate strategies (0/1/2/3 stops)
- Outputs:
  - recommended compounds per stint
  - predicted total race time
  - delta vs best (and comparisons)

It explicitly includes pit stop time loss so you can evaluate if extra stops are worth it.

In [1]:
import re
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

## 1) Config

In [2]:
DATA_DIR = Path("openf1_data")
assert DATA_DIR.exists(), f"Missing openf1_data directory: {DATA_DIR.resolve()}"

MODEL_FILE = "lap_time_model.joblib"
ENC_FILE = "strategy_sim_encoders.joblib"

# Inference input files (separate context + weather forecast)
RACE_CONTEXT_FILE = "race_context_input.csv"
FORECAST_FILE = "forecast_weather.csv"

# candidate compounds
VALID_COMPOUNDS = ["SOFT", "MEDIUM", "HARD"]

# fallback/default pit loss (seconds) if unavailable for a track
DEFAULT_PIT_LOSS_SEC = 21.0

## 2) Utility functions

In [3]:
def parse_session_key(path: Path):
    m = re.search(r"_session_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else None


def safe_read_csv(path: Path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()


def coerce_seconds_col(df, col):
    if col not in df.columns:
        df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

## 3) Build training table from local laps + stints + weather + drivers

In [4]:
def load_sessions_and_track_map():
    sessions = safe_read_csv(DATA_DIR / "sessions.csv")
    if sessions.empty:
        return {}, {}

    for c in ["session_key", "meeting_key", "session_name", "location", "circuit_short_name"]:
        if c not in sessions.columns:
            sessions[c] = np.nan

    # Optional meeting detail files
    meeting_files = list(DATA_DIR.glob("meetings_meeting_*.csv"))
    m_rows = []
    for f in meeting_files:
        m = safe_read_csv(f)
        if m.empty:
            continue
        for c in ["meeting_key", "meeting_name", "country_name", "location", "circuit_short_name"]:
            if c not in m.columns:
                m[c] = np.nan
        m_rows.append(m[["meeting_key", "meeting_name", "country_name", "location", "circuit_short_name"]].head(1))

    if m_rows:
        meetings = pd.concat(m_rows, ignore_index=True).drop_duplicates("meeting_key")
        sessions = sessions.merge(meetings, on="meeting_key", how="left", suffixes=("", "_m"))
    else:
        sessions["meeting_name"] = np.nan
        sessions["country_name"] = np.nan
        sessions["location_m"] = np.nan
        sessions["circuit_short_name_m"] = np.nan

    sessions["track_name"] = (
        sessions.get("circuit_short_name", pd.Series([np.nan]*len(sessions)))
        .fillna(sessions.get("circuit_short_name_m", pd.Series([np.nan]*len(sessions))))
        .fillna(sessions.get("location", pd.Series([np.nan]*len(sessions))))
        .fillna(sessions.get("location_m", pd.Series([np.nan]*len(sessions))))
        .fillna(sessions.get("meeting_name", pd.Series([np.nan]*len(sessions))))
        .fillna(sessions.get("country_name", pd.Series([np.nan]*len(sessions))))
        .fillna(sessions["meeting_key"].astype(str))
    )

    session_to_track = dict(zip(sessions["session_key"], sessions["track_name"]))
    session_to_name = dict(zip(sessions["session_key"], sessions["session_name"].astype(str)))
    return session_to_track, session_to_name


def make_training_rows(max_sessions=None):
    session_to_track, session_to_name = load_sessions_and_track_map()

    laps_files = sorted(DATA_DIR.glob("laps_session_*.csv"))
    if max_sessions is not None:
        laps_files = laps_files[:max_sessions]

    rows = []

    for lf in laps_files:
        sid = parse_session_key(lf)
        if sid is None:
            continue

        # Optionally focus on race sessions only (recommended)
        sname = session_to_name.get(sid, "")
        if "Race" not in sname and "RACE" not in sname:
            continue

        laps = safe_read_csv(lf)
        if laps.empty:
            continue

        # We need driver_number, lap_number, lap_duration
        required_lap = ["driver_number", "lap_number", "lap_duration"]
        for c in required_lap:
            if c not in laps.columns:
                laps[c] = np.nan
        laps = coerce_seconds_col(laps, "lap_duration")
        laps["lap_number"] = pd.to_numeric(laps["lap_number"], errors="coerce")
        laps = laps.dropna(subset=["driver_number", "lap_number", "lap_duration"])
        if laps.empty:
            continue

        # Filter out safety car/outlier laps aggressively
        laps = laps[(laps["lap_duration"] > 45) & (laps["lap_duration"] < 180)]
        if laps.empty:
            continue

        # Stints for compound + tire age estimate
        stints = safe_read_csv(DATA_DIR / f"stints_session_{sid}.csv")
        if stints.empty:
            continue
        for c in ["driver_number", "compound", "lap_start", "lap_end"]:
            if c not in stints.columns:
                stints[c] = np.nan
        stints["lap_start"] = pd.to_numeric(stints["lap_start"], errors="coerce")
        stints["lap_end"] = pd.to_numeric(stints["lap_end"], errors="coerce")

        # Drivers for team
        dr = safe_read_csv(DATA_DIR / f"drivers_session_{sid}.csv")
        if dr.empty:
            continue
        for c in ["driver_number", "team_name"]:
            if c not in dr.columns:
                dr[c] = np.nan
        dr_map = dr[["driver_number", "team_name"]].dropna().drop_duplicates()

        # Starting grid for position
        grid = safe_read_csv(DATA_DIR / f"starting_grid_session_{sid}.csv")
        if not grid.empty:
            pos_col = "position" if "position" in grid.columns else ("grid_position" if "grid_position" in grid.columns else None)
            if pos_col is not None and "driver_number" in grid.columns:
                grid = grid[["driver_number", pos_col]].copy()
                grid.columns = ["driver_number", "starting_position"]
                grid["starting_position"] = pd.to_numeric(grid["starting_position"], errors="coerce")
            else:
                grid = pd.DataFrame(columns=["driver_number", "starting_position"])
        else:
            grid = pd.DataFrame(columns=["driver_number", "starting_position"])

        # Weather summary for session
        w = safe_read_csv(DATA_DIR / f"weather_session_{sid}.csv")
        for c in ["air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]:
            if c not in w.columns:
                w[c] = np.nan
            w[c] = pd.to_numeric(w[c], errors="coerce")

        weather_feat = {
            "air_temp_mean": float(w["air_temperature"].mean()) if not w.empty else np.nan,
            "track_temp_mean": float(w["track_temperature"].mean()) if not w.empty else np.nan,
            "humidity_mean": float(w["humidity"].mean()) if not w.empty else np.nan,
            "rain_minutes_ratio": float((w["rainfall"].fillna(0) > 0).mean()) if not w.empty else 0.0,
            "wind_speed_mean": float(w["wind_speed"].mean()) if not w.empty else np.nan,
        }

        # Build quick lookup for stint by (driver, lap)
        stint_rows = []
        for _, s in stints.iterrows():
            dn = s.get("driver_number", np.nan)
            comp = s.get("compound", np.nan)
            ls = s.get("lap_start", np.nan)
            le = s.get("lap_end", np.nan)
            if pd.isna(dn) or pd.isna(ls) or pd.isna(le) or pd.isna(comp):
                continue
            stint_rows.append((int(dn), str(comp).upper(), int(ls), int(le)))

        # Pre-build team and grid lookup
        team_lookup = dict(zip(dr_map["driver_number"], dr_map["team_name"]))
        grid_lookup = dict(zip(grid["driver_number"], grid["starting_position"]))

        for _, lap in laps.iterrows():
            dn = lap["driver_number"]
            ln = int(lap["lap_number"])
            lap_time = float(lap["lap_duration"])

            team = team_lookup.get(dn, "UNKNOWN_TEAM")
            start_pos = grid_lookup.get(dn, np.nan)
            if pd.isna(start_pos):
                start_pos = 10.0

            comp = None
            tire_age = None
            for dn2, c2, ls, le in stint_rows:
                if int(dn) == dn2 and ls <= ln <= le:
                    comp = c2
                    tire_age = ln - ls
                    break
            if comp is None:
                continue
            if comp not in VALID_COMPOUNDS:
                continue

            rows.append({
                "session_key": sid,
                "track_name": str(session_to_track.get(sid, "UNKNOWN_TRACK")),
                "team_name": str(team),
                "starting_position": float(start_pos),
                "compound": comp,
                "lap_number": ln,
                "tire_age": float(max(tire_age, 0)),
                "lap_time": lap_time,
                **weather_feat,
            })

    return pd.DataFrame(rows)


train_df = make_training_rows(max_sessions=None)
print("Training rows:", len(train_df))
train_df.head()

Training rows: 7728


,session_key,track_name,team_name,starting_position,compound,lap_number,tire_age,lap_time,air_temp_mean,track_temp_mean,humidity_mean,rain_minutes_ratio,wind_speed_mean
0,10033,Miami,Kick Sauber,10.0,MEDIUM,1,0.0,93.275,26.573154,38.690604,63.020134,0.033557,1.04698
1,10033,Miami,Aston Martin,10.0,HARD,1,0.0,93.165,26.573154,38.690604,63.020134,0.033557,1.04698
2,10033,Miami,Aston Martin,10.0,MEDIUM,1,0.0,93.004,26.573154,38.690604,63.020134,0.033557,1.04698
3,10033,Miami,Ferrari,10.0,HARD,1,0.0,92.636,26.573154,38.690604,63.020134,0.033557,1.04698
4,10033,Miami,Williams,10.0,MEDIUM,1,0.0,92.076,26.573154,38.690604,63.020134,0.033557,1.04698


## 4) Train lap-time regressor

In [5]:
FEATURE_CAT = ["team_name", "track_name", "compound"]
FEATURE_NUM = [
    "starting_position",
    "lap_number",
    "tire_age",
    "air_temp_mean",
    "track_temp_mean",
    "humidity_mean",
    "rain_minutes_ratio",
    "wind_speed_mean",
]
TARGET = "lap_time"

df = train_df.copy()
df = df.dropna(subset=[TARGET])

for c in FEATURE_CAT:
    df[c] = df[c].fillna("UNKNOWN").astype(str)
for c in FEATURE_NUM:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].median())

encoders = {}
for c in FEATURE_CAT:
    le = LabelEncoder()
    df[c + "_enc"] = le.fit_transform(df[c])
    encoders[c] = le

X = df[[c + "_enc" for c in FEATURE_CAT] + FEATURE_NUM]
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=18,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)
print(f"Lap-time model MAE: {mae:.3f} sec")

joblib.dump(model, MODEL_FILE)
joblib.dump({
    "encoders": encoders,
    "feature_cat": FEATURE_CAT,
    "feature_num": FEATURE_NUM,
}, ENC_FILE)
print("Saved model and encoders")

Lap-time model MAE: 0.997 sec
Saved model and encoders


## 5) Estimate track-specific pit loss from historical data (fallback to default)

In [6]:
def estimate_track_pit_loss_map():
    session_to_track, session_to_name = load_sessions_and_track_map()
    pit_map = {}

    laps_files = sorted(DATA_DIR.glob("laps_session_*.csv"))
    for lf in laps_files:
        sid = parse_session_key(lf)
        if sid is None:
            continue
        sname = session_to_name.get(sid, "")
        if "Race" not in sname and "RACE" not in sname:
            continue

        laps = safe_read_csv(lf)
        pits = safe_read_csv(DATA_DIR / f"pit_session_{sid}.csv")
        if laps.empty or pits.empty:
            continue

        for c in ["driver_number", "lap_number", "lap_duration"]:
            if c not in laps.columns:
                laps[c] = np.nan
        laps["lap_number"] = pd.to_numeric(laps["lap_number"], errors="coerce")
        laps["lap_duration"] = pd.to_numeric(laps["lap_duration"], errors="coerce")
        laps = laps.dropna(subset=["driver_number", "lap_number", "lap_duration"])

        pit_lap_col = "lap_number" if "lap_number" in pits.columns else None
        if pit_lap_col is None or "driver_number" not in pits.columns:
            continue
        pits[pit_lap_col] = pd.to_numeric(pits[pit_lap_col], errors="coerce")
        pits = pits.dropna(subset=["driver_number", pit_lap_col])

        track = str(session_to_track.get(sid, "UNKNOWN_TRACK"))
        extra = []

        for _, p in pits.iterrows():
            dn = p["driver_number"]
            pl = int(p[pit_lap_col])

            d = laps[laps["driver_number"] == dn].sort_values("lap_number")
            cur = d[d["lap_number"] == pl]
            prev3 = d[(d["lap_number"] >= pl-3) & (d["lap_number"] < pl)]
            if cur.empty or prev3.empty:
                continue

            pit_lap_time = float(cur["lap_duration"].iloc[0])
            base = float(prev3["lap_duration"].median())
            add = pit_lap_time - base
            if 5 < add < 60:
                extra.append(add)

        if extra:
            pit_map[track] = float(np.median(extra))

    return pit_map


track_pit_loss = estimate_track_pit_loss_map()
print("Estimated pit loss for tracks:", len(track_pit_loss))
list(track_pit_loss.items())[:10]

Estimated pit loss for tracks: 7


[('Miami', 25.82),
 ('Spa-Francorchamps', 5.533500000000004),
 ('Spielberg', 6.117999999999995),
 ('Silverstone', 10.587999999999994),
 ('Interlagos', 37.815),
 ('Baku', 13.469999999999999),
 ('Imola', 27.453000000000003)]

## 6) Create example inference input files

In [7]:
race_context = pd.read_csv(RACE_CONTEXT_FILE)
forecast = pd.read_csv(FORECAST_FILE)
print("Wrote example inference input files")

Wrote example inference input files


## 7) Strategy simulation engine

In [8]:
def encode_with_unknown(le, value):
    value = str(value)
    if value in set(le.classes_):
        return int(le.transform([value])[0])
    return 0


def forecast_to_summary(f):
    req = ["air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]
    miss = [c for c in req if c not in f.columns]
    if miss:
        raise ValueError(f"{FORECAST_FILE} missing columns {miss}")

    ff = f.copy()
    for c in req:
        ff[c] = pd.to_numeric(ff[c], errors="coerce")
    ff = ff.dropna(subset=req)
    if ff.empty:
        raise ValueError("No valid forecast rows")

    return {
        "air_temp_mean": float(ff["air_temperature"].mean()),
        "track_temp_mean": float(ff["track_temperature"].mean()),
        "humidity_mean": float(ff["humidity"].mean()),
        "rain_minutes_ratio": float((ff["rainfall"] > 0).mean()),
        "wind_speed_mean": float(ff["wind_speed"].mean()),
    }


def split_laps(total_laps, n_stints):
    # near-even split; can be replaced by optimization later
    base = total_laps // n_stints
    rem = total_laps % n_stints
    out = []
    for i in range(n_stints):
        out.append(base + (1 if i < rem else 0))
    return out


def build_candidate_strategies(starting_compound, max_stops=3):
    candidates = []
    start = str(starting_compound).upper()
    if start not in VALID_COMPOUNDS:
        start = "MEDIUM"

    for stops in range(0, max_stops + 1):
        n_stints = stops + 1
        if n_stints == 1:
            candidates.append([start])
            continue

        # choose compounds for subsequent stints
        for seq in product(VALID_COMPOUNDS, repeat=stops):
            full = [start] + list(seq)
            # optional simple rule: avoid identical all-stint compound for >1 stops
            if len(set(full)) == 1 and stops >= 1:
                continue
            candidates.append(full)
    return candidates


def predict_lap_time(model, enc, row_dict):
    feat_cat = enc["feature_cat"]
    feat_num = enc["feature_num"]
    le_map = enc["encoders"]

    r = row_dict.copy()
    for c in feat_cat:
        r[c + "_enc"] = encode_with_unknown(le_map[c], r[c])

    x = pd.DataFrame([{**{c + "_enc": r[c + "_enc"] for c in feat_cat}, **{c: r[c] for c in feat_num}}])
    return float(model.predict(x)[0])


def simulate_strategy(model, enc, track_pit_loss_map, race_context, weather_summary, compounds):
    total_laps = int(race_context["race_laps"])
    n_stints = len(compounds)
    stint_lengths = split_laps(total_laps, n_stints)

    pit_loss = float(track_pit_loss_map.get(str(race_context["track_name"]), DEFAULT_PIT_LOSS_SEC))
    total_time = 0.0

    lap_global = 1
    detailed_stints = []

    for i, (comp, stint_len) in enumerate(zip(compounds, stint_lengths), start=1):
        stint_time = 0.0
        for lap_in_stint in range(1, stint_len + 1):
            feat = {
                "team_name": race_context["team_name"],
                "track_name": race_context["track_name"],
                "compound": comp,
                "starting_position": float(race_context["starting_position"]),
                "lap_number": float(lap_global),
                "tire_age": float(lap_in_stint - 1),
                **weather_summary,
            }
            lap_t = predict_lap_time(model, enc, feat)
            stint_time += lap_t
            lap_global += 1

        total_time += stint_time
        detailed_stints.append({
            "stint": i,
            "compound": comp,
            "laps": stint_len,
            "predicted_stint_time_sec": stint_time,
        })

        # add pit loss between stints
        if i < n_stints:
            total_time += pit_loss

    return {
        "compounds": compounds,
        "stops": n_stints - 1,
        "pit_loss_per_stop_sec": pit_loss,
        "predicted_total_time_sec": total_time,
        "stints": detailed_stints,
    }


def format_seconds(sec):
    m = int(sec // 60)
    s = sec - 60*m
    return f"{m}:{s:06.3f}"

## 8) Run recommendation

In [9]:
lap_model = joblib.load(MODEL_FILE)
enc = joblib.load(ENC_FILE)

race_context = pd.read_csv(RACE_CONTEXT_FILE).iloc[0].to_dict()
forecast = pd.read_csv(FORECAST_FILE)
weather_summary = forecast_to_summary(forecast)

# Normalize context
race_context["team_name"] = str(race_context.get("team_name", "UNKNOWN_TEAM"))
race_context["track_name"] = str(race_context.get("track_name", "UNKNOWN_TRACK"))
race_context["starting_compound"] = str(race_context.get("starting_compound", "MEDIUM")).upper()
race_context["starting_position"] = float(race_context.get("starting_position", 10))
race_context["race_laps"] = int(race_context.get("race_laps", 60))

cands = build_candidate_strategies(race_context["starting_compound"], max_stops=3)
results = []

for c in cands:
    sim = simulate_strategy(lap_model, enc, track_pit_loss, race_context, weather_summary, c)
    results.append(sim)

res_df = pd.DataFrame([
    {
        "strategy": " -> ".join(r["compounds"]),
        "stops": r["stops"],
        "predicted_total_time_sec": r["predicted_total_time_sec"],
        "pit_loss_per_stop_sec": r["pit_loss_per_stop_sec"],
    }
    for r in results
]).sort_values("predicted_total_time_sec").reset_index(drop=True)

best_time = float(res_df.loc[0, "predicted_total_time_sec"])
res_df["delta_to_best_sec"] = res_df["predicted_total_time_sec"] - best_time

print("\n--- TOP 10 STRATEGIES ---")
show = res_df.head(10).copy()
show["predicted_total_time"] = show["predicted_total_time_sec"].apply(format_seconds)
show["delta_to_best"] = show["delta_to_best_sec"].apply(lambda x: f"+{x:.3f}s")
display(show[["strategy", "stops", "predicted_total_time", "delta_to_best", "pit_loss_per_stop_sec"]])

best_strategy = results[int(res_df.index[0])]
best_compounds = res_df.loc[0, "strategy"]

print("\n=== RECOMMENDED STRATEGY ===")
print("Team:", race_context["team_name"])
print("Track:", race_context["track_name"])
print("Start pos:", race_context["starting_position"])
print("Starting tire:", race_context["starting_compound"])
print("Recommended sequence:", best_compounds)
print("Stops:", int(res_df.loc[0, "stops"]))
print("Predicted total time:", format_seconds(float(res_df.loc[0, "predicted_total_time_sec"])))
print("Assumed pit loss per stop:", f"{float(res_df.loc[0, 'pit_loss_per_stop_sec']):.2f}s")

print("\nStint breakdown (best):")
for i, st in enumerate(results[0]["stints"], start=1):
    print(f"  Stint {i}: {st['compound']} | laps={st['laps']} | predicted={format_seconds(st['predicted_stint_time_sec'])}")


--- TOP 10 STRATEGIES ---


,strategy,stops,predicted_total_time,delta_to_best,pit_loss_per_stop_sec
0,MEDIUM -> SOFT,1,99:57.488,+0.000s,27.453
1,MEDIUM -> HARD,1,99:59.397,+1.909s,27.453
2,MEDIUM,0,100:12.729,+15.241s,27.453
3,MEDIUM -> HARD -> SOFT,2,100:16.012,+18.524s,27.453
4,MEDIUM -> HARD -> MEDIUM,2,100:18.998,+21.510s,27.453
5,MEDIUM -> SOFT -> SOFT,2,100:23.223,+25.735s,27.453
6,MEDIUM -> MEDIUM -> SOFT,2,100:25.617,+28.129s,27.453
7,MEDIUM -> SOFT -> MEDIUM,2,100:26.209,+28.721s,27.453
8,MEDIUM -> HARD -> HARD,2,100:33.239,+35.752s,27.453
9,MEDIUM -> SOFT -> HARD,2,100:40.450,+42.962s,27.453



=== RECOMMENDED STRATEGY ===
Team: Ferrari
Track: Imola
Start pos: 4.0
Starting tire: MEDIUM
Recommended sequence: MEDIUM -> SOFT
Stops: 1
Predicted total time: 99:57.488
Assumed pit loss per stop: 27.45s

Stint breakdown (best):
  Stint 1: MEDIUM | laps=63 | predicted=100:12.729


## 9) Notes / next improvements

- Add explicit safety-car probability and branch scenarios (SC vs no-SC).
- Replace even lap split with optimization over pit windows (dynamic programming).
- Add traffic model (position-dependent dirty air/overtake penalty).
- Add mandatory-two-compound race-rule constraints when applicable.
- Fit track-specific pit loss from cleaner pit in/out events.